# Chains in LangChain

이번 수업은 LangChain의 가장 중요한 핵심 빌딩 블록, 즉 **Chain**을 배워봅니다. Chain은 일반적으로 LLM, **대규모 언어 모델을 프롬프트와 결합**하며, 이 빌딩 블록을 사용하면, 이러한 빌딩 블록을 함께 배치하여 텍스트나 기타 데이터에 대한 일련의 작업을 수행할 수도 있습니다.


## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [2]:
import openai
import os
from dotenv import load_dotenv, find_dotenv
from getpass import getpass

# 사용할 API Key 이름
API_KEY_NAME = "OPENAI_API_KEY"

if find_dotenv():
    _ = load_dotenv(find_dotenv())
    openai.api_key = os.getenv(API_KEY_NAME)
else:
    openai.api_key = getpass(f"Enter {API_KEY_NAME}: ")
    os.environ[API_KEY_NAME] = openai.api_key

print(f"{API_KEY_NAME} loaded successfully.")

import warnings
warnings.filterwarnings('ignore')

OPENAI_API_KEY loaded successfully.


In [3]:
import pandas as pd
url = "https://raw.githubusercontent.com/junji64/LangChain/main/Data.csv"
df = pd.read_csv(url)

In [4]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

LLMChain은 LLM과 프롬프트의 조합일 뿐입니다.

In [6]:
!pip install langchain-classic langchain-community langchain-openai

Defaulting to user installation because normal site-packages is not writeable


In [7]:
from langchain_openai import ChatOpenAI
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

In [8]:
llm = ChatOpenAI(temperature=0.9)

In [9]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

In [10]:
chain = LLMChain(llm=llm, prompt=prompt)

C:\Users\User\AppData\Local\Temp\ipykernel_19000\1305865249.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


In [11]:
product = "Queen Size Sheet Set"
chain.run(product)

C:\Users\User\AppData\Local\Temp\ipykernel_19000\550859008.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  chain.run(product)


'"Regal Comfort Co."'

## SequentialChain

Sequential chain 은 또 다른 유형의 체인입니다. 아이디어는 한 체인의 출력이 다음 체인의 입력인 여러 체인을 결합하는 것입니다.

Sequential chain 에는 두 가지 유형이 있습니다.

1. SimpleSequencialChain : Single input/output

2. SequentialChain : multiple inputs/outputs

## SimpleSequentialChain

<img src="./SSC.PNG" width=400>

In [13]:
from langchain_classic.chains import SimpleSequentialChain

In [14]:
llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [15]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [16]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True)

In [17]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
"Royal Comfort Linens"
Royal Comfort Linens offers luxurious and high-quality bedding and bath products to elevate your home with style and comfort.

> Finished chain.


'Royal Comfort Linens offers luxurious and high-quality bedding and bath products to elevate your home with style and comfort.'

## SequentialChain

<img src="./SC.PNG" width=400>

In [19]:
from langchain_classic.chains import SequentialChain

In [20]:
llm = ChatOpenAI(temperature=0.9)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="English_Review"
                    )

In [21]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )

In [22]:
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )

In [23]:

# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )

In [24]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary","language","followup_message"],
    verbose=True
)

In [25]:
review = df.Review[6]
overall_chain(review)



> Entering new SequentialChain chain...


C:\Users\User\AppData\Local\Temp\ipykernel_19000\1878728458.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  overall_chain(review)



> Finished chain.


{'Review': 'Está lu bonita calienta muy rápido, es muy funcional, solo falta ver cuánto dura, solo llevo 3 días en funcionamiento.',
 'English_Review': 'It is very beautiful, heats up very quickly, is very functional, just need to see how long it lasts, only been in operation for 3 days.',
 'summary': 'This product is beautiful, functional, and heats up quickly, but it is too early to tell how long it will last as it has only been used for 3 days.',
 'language': 'This is written in Spanish.',
 'followup_message': 'Gracias por compartir tu opinión sobre el producto. Es genial saber que es hermoso, funcional y se calienta rápidamente. Sin embargo, es comprensible que aún no puedas determinar su durabilidad después de solo 3 días de uso. Espero que sigas disfrutando del producto y que nos mantengas actualizados sobre su rendimiento a largo plazo. ¡Gracias de nuevo por compartir tus comentarios!'}

## Router Chain

지금까지 우리는 LLMChain과 SequentialChain을 다루었습니다.
하지만 좀 더 복잡한 작업을 수행하고 싶다면 어떻게 해야 할까요?
매우 일반적이지만 기본적인 작업은 **해당 입력이 정확히 무엇인지에 따라 입력을 적절한 체인으로 라우팅하는 것**입니다.
이것을 상상하는 좋은 방법은 특정 유형의 입력에 특화된 여러 하위 체인이 있는 경우 **먼저 전달할 하위 체인을 결정**한 다음, **해당 체인에 전달**하는 **라우터 체인**을 가질 수 있다는 것입니다.

<img src="./RC.PNG" width=400>

In [27]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [28]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [29]:
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain_classic.chains.router.multi_prompt import MultiPromptChain

In [30]:
llm = ChatOpenAI(temperature=0)

In [31]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [32]:
destinations

['physics: Good for answering questions about physics',
 'math: Good for answering math questions',
 'History: Good for answering history questions',
 'computer science: Good for answering computer science questions']

In [33]:
destinations_str

'physics: Good for answering questions about physics\nmath: Good for answering math questions\nHistory: Good for answering history questions\ncomputer science: Good for answering computer science questions'

In [34]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or “default”
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "default" if the input is not \
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [35]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [36]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [37]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, 
                         verbose=True)

C:\Users\User\AppData\Local\Temp\ipykernel_19000\757166367.py:1: LangChainDeprecationWarning: The class `MultiPromptChain` was deprecated in LangChain 0.2.12 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build routing logic with `create_agent` (e.g. with subagents or prompt-selection middleware). See https://docs.langchain.com/oss/python/langchain/agents
  chain = MultiPromptChain(router_chain=router_chain,


In [38]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation refers to the electromagnetic radiation emitted by a perfect black body, which is an idealized physical body that absorbs all incident electromagnetic radiation and emits radiation at all frequencies. The radiation emitted by a black body depends only on its temperature and follows a specific distribution known as Planck's law. This type of radiation is important in understanding concepts such as thermal radiation and the behavior of objects at different temperatures."

In [39]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [40]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
None: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA carries the genetic information that determines the characteristics and functions of an organism. DNA contains the instructions for building and maintaining an organism, including the proteins that are essential for cell structure and function. This genetic information is passed down from parent to offspring and is essential for the growth, development, and functioning of all cells in the body. Having DNA in every cell ensures that the organism can carry out its biological processes and maintain its overall structure and function.'